# 과제 1.

- OpenAI 클라이언트와 실제 영화 API를 사용하여 Movie Expert Agent를 구축하세요.


In [28]:
# LLM에게 역할을 부여하는 프롬프트 작성

ROLE_PROMPT = """
당신은 영화 추천 전문가입니다. 사용자의 영화 추천 질문에 간결하게 대답해야 합니다.

### Instruction

- **사용자가 좋아하는 장르를 기억해야 합니다.**

- **이미 시청한 영화를 기억해야 합니다.**

- **대화 기록을 기반으로 개인화된 추천을 제공해야 합니다.**

- **도구를 호출할 때는 하나의 질문에 하나의 도구만 호출해야 합니다.**
"""

In [29]:
import openai, requests, json

client = openai.OpenAI()

url = "https://nomad-movies.nomadcoders.workers.dev"

messages = [{"role": "system", "content": ROLE_PROMPT}]

In [30]:
# 함수 목록


def is_status_200(response):
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code}")


def get_popular_movies():
    """/movies에서 인기 영화를 가져옵니다."""
    response = requests.get(f"{url}/movies")
    return is_status_200(response)


def get_movie_details(id):
    """/movies/id에서 영화 정보를 가져옵니다."""
    response = requests.get(f"{url}/movies/{id}")

    return is_status_200(response)


def get_movie_credits(id):
    """/movies/id/credits에서 출연진 및 제작진을 가져옵니다."""
    response = requests.get(f"{url}/movies/{id}/credits")
    return is_status_200(response)


def get_movie_similar(id):
    """/movies/id/similar에서 유사한 영화를 조회합니다."""
    response = requests.get(f"{url}/movies/{id}/similar")
    return is_status_200(response)

In [31]:
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_movie_similar": get_movie_similar,
}

In [32]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "/movies에서 인기 영화를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {},
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "/movies/id에서 영화 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "number",
                        "description": "특정 영화의 id",
                    }
                },
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "/movies/id/credits에서 출연진 및 제작진을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "number",
                        "description": "특정 영화의 id",
                    }
                },
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_similar",
            "description": "/movies/id/similar에서 유사한 영화를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "number",
                        "description": "특정 영화의 id",
                    }
                },
                "additionalProperties": False,
            },
        },
    },
]

In [37]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            try:
                parsed_args = json.loads(arguments)
            except json.JSONDecodeError:
                parsed_args = {}

            if parsed_args:
                args_str = ", ".join(f"{k}={v}" for k, v in parsed_args.items())
                print(f"Agent: [{function_name}({args_str}) 호출]")
            else:
                print(f"Agent: [{function_name}() 호출]")

            function_to_run = FUNCTION_MAP.get(function_name)
            result = function_to_run(**parsed_args)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(result),
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content[:50]}...")
        print("-" * 50)


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [38]:
while True:
    message = input("LLM에게 보낼 메세지를 보내세요.")

    if message == "종료":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 지금 인기있는 영화 알려주세요
Agent: [get_popular_movies() 호출]
Agent: 현재 인기 있는 영화는 다음과 같습니다:

1. **Shelter**
   - 개봉일: 2...
--------------------------------------------------
User: Shelter에 대해 더 알려주세요
Agent: [get_movie_details(id=1290821) 호출]
Agent: **Shelter**에 대한 상세 정보는 다음과 같습니다:

- **제목:** Shelte...
--------------------------------------------------
User: Shelter의 출연진에 대해 알려주세요
Agent: [get_movie_credits(id=1290821) 호출]
Agent: **Shelter**의 주요 출연진은 다음과 같습니다:

1. **제이슨 스타뎀 (Jaso...
--------------------------------------------------
User: 비슷한 영화 추천해주세요
Agent: [get_movie_similar(id=1290821) 호출]
Agent: **Shelter**와 비슷한 영화 추천 목록은 다음과 같습니다:

1. **Rosevil...
--------------------------------------------------
